In [ ]:
# Hosted D2L setup: fetch the exact helper module used to build this notebook.
from pathlib import Path
from urllib.request import urlretrieve
from importlib.metadata import PackageNotFoundError, version
import importlib.util, os, subprocess, sys

required = ['numpy', 'pandas', 'matplotlib', 'requests', 'scipy', 'pillow', 'regex', 'jax', 'jaxlib', 'flax', 'optax', 'orbax-checkpoint', 'tensorflow', 'protobuf', 'ml-dtypes', 'safetensors']
imports = {'pillow': 'PIL', 'orbax-checkpoint': 'orbax', 'protobuf': 'google.protobuf', 'ml-dtypes': 'ml_dtypes'}
pinned = {'jax': ('0.10.2', 'jax==0.10.2', 'jax[cuda12]==0.10.2', 'exact'), 'jaxlib': ('0.10.2', 'jaxlib==0.10.2', 'jaxlib==0.10.2', 'exact'), 'flax': ('0.12.7', 'flax==0.12.7', 'flax==0.12.7', 'exact'), 'optax': ('0.2.8', 'optax==0.2.8', 'optax==0.2.8', 'exact'), 'orbax-checkpoint': ('0.12.0', 'orbax-checkpoint==0.12.0', 'orbax-checkpoint==0.12.0', 'exact'), 'safetensors': ('0.8.0', 'safetensors==0.8.0', 'safetensors==0.8.0', 'exact')}
fallbacks = {'tensorflow': 'tensorflow==2.21.0', 'protobuf': 'protobuf==7.34.1', 'ml-dtypes': 'ml-dtypes==0.5.4'}
device = os.environ.get("D2L_HOSTED_DEVICE", "auto").lower()
if device not in ("auto", "cpu", "gpu"):
    raise ValueError(f"Invalid D2L_HOSTED_DEVICE={device!r}")
if device == "auto":
    try:
        gpu = (Path("/dev/nvidia0").exists() or
               subprocess.run(["nvidia-smi", "-L"], capture_output=True,
                              timeout=5).returncode == 0)
    except (FileNotFoundError, subprocess.SubprocessError):
        gpu = False
else:
    gpu = device == "gpu"
if not gpu:
    os.environ.setdefault("CUDA_VISIBLE_DEVICES", "-1")
    os.environ.setdefault("JAX_PLATFORMS", "cpu")
tensorflow_version = None
if 'jax' in ("tensorflow", "jax"):
    try:
        tensorflow_version = version("tensorflow")
    except PackageNotFoundError:
        pass
# Colab's CPU image currently carries a CUDA-enabled TensorFlow wheel. Its
# first ordinary tensor operation probes CUDA and emits an error-level cuInit
# diagnostic. JAX notebooks also use TensorFlow for data loading, so overlay
# the matching CPU build in both CPU variants. Keep the provider's
# ``tensorflow`` distribution metadata: other preinstalled Colab packages
# depend on that distribution name, while both wheels expose the same module.
if not gpu and 'jax' in ("tensorflow", "jax"):
    try:
        tensorflow_cpu_version = version("tensorflow-cpu")
    except PackageNotFoundError:
        tensorflow_cpu_version = None
    if (tensorflow_version is not None and
            tensorflow_cpu_version != tensorflow_version):
        subprocess.check_call([
            sys.executable, "-m", "pip", "install", "-q", "--no-deps",
            f"tensorflow-cpu=={tensorflow_version}",
        ])
if "tf-keras" in fallbacks and tensorflow_version is not None:
    fallbacks["tf-keras"] = f"tf-keras=={tensorflow_version}"
missing = []
for package in required:
    if package in pinned:
        wanted, cpu_requirement, gpu_requirement, match = pinned[package]
        requirement = gpu_requirement if gpu else cpu_requirement
        try:
            installed = version(package)
        except PackageNotFoundError:
            installed = None
        actual = (installed.split("+", 1)[0]
                  if installed is not None and match == "public" else installed)
        if actual != wanted:
            missing.append(requirement)
    elif importlib.util.find_spec(imports.get(package, package)) is None:
        missing.append(fallbacks.get(package, package))
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])

mismatched = []
for package, (wanted, _, _, match) in pinned.items():
    try:
        installed = version(package)
    except PackageNotFoundError:
        installed = None
    actual = (installed.split("+", 1)[0]
              if installed is not None and match == "public" else installed)
    if actual != wanted:
        mismatched.append(f"{package}={installed!r} (expected {wanted})")
if mismatched:
    raise RuntimeError("Hosted runtime setup failed: " + ", ".join(mismatched))

root = Path(".d2l-hosted") / "5d733eab198ad58f23ceee3f1550014385366ece"
package = root / "d2l"
package.mkdir(parents=True, exist_ok=True)
base = "https://raw.githubusercontent.com/smolix/d2l-neu/5d733eab198ad58f23ceee3f1550014385366ece/d2l"
for name in ('__init__.py', 'jax.py', 'nnx_resnet.py'):
    target = package / name
    if not target.exists():
        urlretrieve(f"{base}/{name}", target)
if str(root.resolve()) not in sys.path:
    sys.path.insert(0, str(root.resolve()))
pythonpath = os.environ.get("PYTHONPATH", "").split(os.pathsep)
if str(root.resolve()) not in pythonpath:
    os.environ["PYTHONPATH"] = os.pathsep.join(
        [str(root.resolve()), *[entry for entry in pythonpath if entry]]
    )


# Saving, Loading, and Pretrained Weights

A trained network is two separate things kept in two separate places. The
*code* is the class you wrote: its layers, its `forward` pass, the config that
sized it. The *model state* is the tensors that training filled in: weights,
biases, and running statistics. Optimizer moments, random-number streams, and
the step belong to the wider *training state*. A weight file saves model state;
a resumable checkpoint saves full training state. The code
stays in your source repository, under version control, exactly like any other
Python. To bring a model back you need both halves: run the code to rebuild an
empty network, then pour the saved state into it.

This split explains most of what follows. It is why a checkpoint cannot
resurrect a model on its own, why the config object from
that section belongs *inside* the checkpoint, and why the
format that stores the state matters once you start sharing files with people who
do not have your code.

In [ ]:
import json
import os
import struct
from dataclasses import asdict, dataclass
import jax
from jax import numpy as jnp
from flax import nnx
import optax
import orbax.checkpoint as ocp
from safetensors.flax import load_file, save_file
from d2l import jax as d2l
from d2l.nnx_resnet import ResNet50

## State, Not Code

The state of a network is a tree of typed, named variables introduced in
that section. Before we save a whole model, the warm-up is that
`jnp.save` and `jnp.load` work on any array, and, through NumPy's pickle
fallback, on the dicts that hold them.

In [ ]:
x = jnp.arange(4)
jnp.save('tensors.npy', {'x': x, 'y': jnp.zeros(4)})
jnp.load('tensors.npy', allow_pickle=True).item()

A model's parameter state is one such structure. Its paths follow the module
graph (`hidden.kernel`, `output.bias`; Flax calls a weight matrix a `kernel`).
Here is the tree for a small MLP.

In [ ]:
class MLP(nnx.Module):
    def __init__(self, rngs=None):
        rngs = nnx.Rngs(0) if rngs is None else rngs
        self.hidden = nnx.Linear(20, 256, rngs=rngs)
        self.output = nnx.Linear(256, 10, rngs=rngs)

    def __call__(self, x):
        return self.output(nnx.relu(self.hidden(x)))

net = MLP()
X = jax.random.normal(d2l.get_key(), (2, 20))
Y = net(X)
params = nnx.state(net, nnx.Param)
[(path, tuple(value.shape)) for path, value in params.flat_state()]

Nothing in this dictionary knows it came from a class called `MLP`. The names
and shapes refill a network with a compatible architecture and naming
structure. Renaming an attribute or changing the nesting can break that match
even when the computation is equivalent; the final section shows how to
migrate such keys explicitly.

## safetensors: the Interchange Format

`jnp.save` writes NumPy's `.npy` format. For a single array that is pure data:
a small header with the dtype and shape, then the raw bytes. The warm-up dict is
another matter. NumPy can only store a dict by falling back to Python's
`pickle`, which does not store data so much as a program that *reconstructs*
data, and loading runs that program; that is what `allow_pickle=True` opted
into. For a file you wrote and never let out of your control this is harmless.
For a file you downloaded it is a remote-code-execution vector: any object in
the stream can name a callable for the loader to invoke, which is why NumPy
refuses pickled contents by default (`allow_pickle=False`).

`allow_pickle=False` is a refusal, not a fix: it keeps the loader safe by
declining to load the very files, dicts of named parameters, that model sharing
needs.

safetensors removes the problem at the root by
having no program to run. As the figure lays out byte
by byte, a safetensors file is an 8-byte little-endian integer giving the
header length, then that many bytes of JSON naming each tensor's dtype, shape,
and byte range, then the raw tensor bytes back to back. There is no opcode
stream to interpret, so the format itself carries no executable program.
Loaders must still validate malformed headers, offsets, and allocation sizes.
It is also framework-neutral and memory-mappable, which is why model hubs
commonly prefer it.
Save and reload the MLP's state through it and confirm the round trip is exact.

![The safetensors file as one horizontal byte strip: an 8-byte header length, a JSON header naming each tensor's dtype, shape, and byte offsets, and the raw tensor bytes packed back to back with no gaps, with two of the file's own data_offsets entries traced down to their exact span in the bar.](https://raw.githubusercontent.com/smolix/d2l-neu/notebooks/img/bg-safetensors-layout.svg)

One mismatch to bridge first: safetensors stores a flat mapping from names to
tensors, while NNX state forms a nested pytree. This MLP has only string-keyed
paths, so two small helpers join them with dots and split them on the way back.
For an arbitrary pytree, preserve typed list indices and escape dots in keys,
or restore values against a template instead of using this shortcut.

In [ ]:
def flatten(tree):
    if hasattr(tree, 'flat_state'):
        leaves = tree.flat_state()
    else:
        leaves = jax.tree_util.tree_flatten_with_path(tree)[0]
        leaves = [(tuple(getattr(k, 'key', getattr(k, 'idx', k))
                         for k in path), value)
                  for path, value in leaves]
    return {'.'.join(map(str, path)): value for path, value in leaves}

def unflatten(flat):
    tree = {}
    for name, value in flat.items():
        *parents, leaf = name.split('.')
        node = tree
        for p in parents:
            node = node.setdefault(p, {})
        node[leaf] = value
    return tree

save_file(flatten(params), 'mlp-jax.safetensors')
restored = unflatten(load_file('mlp-jax.safetensors'))
clone = MLP(nnx.Rngs(1))
clone_state = nnx.state(clone, nnx.Param)
nnx.replace_by_pure_dict(clone_state, restored)
nnx.update(clone, clone_state)
exact = jax.tree_util.tree_all(jax.tree_util.tree_map(
    jnp.array_equal, restored, nnx.to_pure_dict(params)))
exact, jnp.array_equal(clone(X), Y)

Because the header is plain JSON at a known offset, you can read it without the
library and see there is no magic to the format.

In [ ]:
with open('mlp-jax.safetensors', 'rb') as f:
    n = struct.unpack('<Q', f.read(8))[0]   # header length, little-endian
    header = json.loads(f.read(n))
header['hidden.kernel']

`jnp.save` keeps its place for your own scratch arrays and quick experiments.
safetensors is what you use to hand a model to anyone else.

## Checkpointing a Training Run

A checkpoint you can resume from holds more than weights. Resuming means picking
up the optimizer where it stopped, and Adam's state is the running first and
second moments of the gradients from that section. Drop them and
the optimizer restarts its momentum from zero, so the first steps after a resume
no longer behave like a continuation. A full checkpoint therefore bundles the
model state, the optimizer state, the RNG state (so data shuffling and dropout
continue the same stream), the step counter, and the config that sizes the model
when you rebuild it. the figure pairs each of those
five compartments with the exact thing it restores on resume.

![A checkpoint file's five compartments, each paired by an arrow with what it restores on resume: model state_dict with weights, optimizer state with momentum and second moments, RNG state with data order and dropout, step with schedule position, and config with architecture.](https://raw.githubusercontent.com/smolix/d2l-neu/notebooks/img/bg-checkpoint-contents.svg)

NNX exposes the model and optimizer state as pytrees, including the optimizer's
step counter. Orbax, the JAX checkpointing library, saves and restores such trees whole: its
`StandardCheckpointer` commits a temporary directory by rename. The demonstration
overwrites a fixed path with `force=True` so the notebook is rerunnable;
production runs should write numbered steps through `CheckpointManager` and
retain the previous completed step. Because these natives
already cover the job, the jax tab defines no helper of its own; the calls below
are the idiom as you would write it in any project. The config rides along as
one more branch of the saved tree, as does the PRNG key, which in JAX is
explicit data rather than hidden global state.

Train a tiny regressor for a hundred steps and checkpoint it. The `Config`
dataclass is what a rebuild reads to size the model, so it travels with the
weights.

In [ ]:
@dataclass
class Config:
    in_dim: int = 20
    hidden: int = 64
    lr: float = 0.05

def build(cfg):
    return nnx.Sequential(
        nnx.Linear(cfg.in_dim, cfg.hidden, rngs=nnx.Rngs(0)), nnx.relu,
        nnx.Linear(cfg.hidden, 1, rngs=nnx.Rngs(1)))

def fresh_state(cfg):
    model = build(cfg)
    optimizer = nnx.Optimizer(model, optax.adam(cfg.lr), wrt=nnx.Param)
    return model, optimizer

cfg = Config()
model = build(cfg)
data = jax.random.normal(d2l.get_key(), (256, cfg.in_dim))
target = (data @ jax.random.normal(d2l.get_key(), (cfg.in_dim, 1))
          + 0.1 * jax.random.normal(d2l.get_key(), (256, 1)))

def loss(model):
    return jnp.mean((model(data) - target) ** 2)

@nnx.jit
def step(model, optimizer):
    l, grads = nnx.value_and_grad(loss)(model)
    optimizer.update(model, grads)
    return l

model, optimizer = fresh_state(cfg)
train_key = jax.random.key(1)
for _ in range(100):
    l = step(model, optimizer)

ckptr = ocp.StandardCheckpointer()
ckptr.save(os.path.abspath('run-jax'),        # orbax wants an absolute path
           {'model': nnx.to_pure_dict(nnx.state(model)),
            'optimizer': nnx.to_pure_dict(nnx.state(optimizer)),
            'rng': train_key,
            'cfg': asdict(cfg)}, force=True)
int(optimizer.step), round(float(loss(model)), 4)

The restore is exact. Corrupt every parameter, load the checkpoint back, and the
loss returns to where it was.

Orbax fills a template with the saved arrays. We then update freshly built NNX
objects from the restored pure dictionaries. Rebuilding first checks that the
architecture still agrees with the checkpoint structure.

In [ ]:
model_state = nnx.state(model)
nnx.update(model, jax.tree.map(lambda p: p + 1.0, model_state))
before = float(loss(model))
template_model, template_optimizer = fresh_state(cfg)
template = {
    'model': nnx.to_pure_dict(nnx.state(template_model)),
    'optimizer': nnx.to_pure_dict(nnx.state(template_optimizer)),
    'rng': jax.random.key(0),
    'cfg': asdict(Config())}
ckpt = ckptr.restore(os.path.abspath('run-jax'), template)
cfg = Config(**ckpt['cfg'])
train_key = ckpt['rng']
restored_model = nnx.state(model)
nnx.replace_by_pure_dict(restored_model, ckpt['model'])
nnx.update(model, restored_model)
restored_optimizer = nnx.state(optimizer)
nnx.replace_by_pure_dict(restored_optimizer, ckpt['optimizer'])
nnx.update(optimizer, restored_optimizer)
after = float(loss(model))
f'perturbed {before:.2f} -> restored {after:.4f}'

Now the reason the optimizer state is in the file. Resume the run two ways from
the same checkpoint: once restoring the optimizer, once with a fresh one holding
only the weights. The network is near its minimum, so the correct continuation
barely moves. A fresh Adam, with its moment estimates reset and its bias
correction starting over, takes an oversized first step and overshoots.

In [ ]:
full, full_opt = fresh_state(cfg)
full_state, full_opt_state = nnx.state(full), nnx.state(full_opt)
nnx.replace_by_pure_dict(full_state, ckpt['model'])
nnx.replace_by_pure_dict(full_opt_state, ckpt['optimizer'])
nnx.update(full, full_state)
nnx.update(full_opt, full_opt_state)
full_losses = []
for _ in range(5):
    l = step(full, full_opt)
    full_losses.append(round(float(l), 4))

fresh, fresh_opt = fresh_state(cfg)
fresh_state_ = nnx.state(fresh)
nnx.replace_by_pure_dict(fresh_state_, ckpt['model'])
nnx.update(fresh, fresh_state_)
fresh_losses = []
for _ in range(5):
    l = step(fresh, fresh_opt)
    fresh_losses.append(round(float(l), 4))

print('full  optimizer:', full_losses)
print('fresh optimizer:', fresh_losses)

The full-state run keeps descending; the weights-only run spikes and has to claw
its way back. That transient is the cost of forgetting the optimizer, and it is
why "just the weights" is not a resumable checkpoint.

For models too large to hold in memory, orbax already works at scale: a
checkpoint is a directory of per-array files rather than one monolith, a restore
can target a device sharding so each accelerator materializes only its own
pieces, and multi-host jobs save and restore in parallel;
that section returns to the machinery when models get that big.

## Loading Weights You Did Not Train

The most common reason to load parameters is that someone else produced them.
You take a network trained on a large dataset and adapt it: keep the learned
feature extractor, replace the final layer for your own labels. The book's NNX
ResNet-50 loader downloads pinned Microsoft ImageNet safetensors from the
Hugging Face Hub and maps them into an NNX model. We use that real artifact,
then reuse the small MLP file to show the key-by-key surgery without burying the
mechanism under a ResNet-sized state tree.

In [ ]:
pretrained_net = ResNet50.from_pretrained()
pretrained_net.fc = nnx.Linear(2048, 10, rngs=nnx.Rngs(2))
print('new pretrained-model head:', pretrained_net.fc.kernel.shape)

class Classifier(nnx.Module):
    def __init__(self, rngs=None):
        rngs = nnx.Rngs(0) if rngs is None else rngs
        self.hidden = nnx.Linear(20, 256, rngs=rngs)
        self.head = nnx.Linear(256, 2, rngs=rngs)  # new head, new name

    def __call__(self, x):
        return self.head(nnx.relu(self.hidden(x)))

classifier = Classifier()
new_params = nnx.state(classifier, nnx.Param)
[(path, tuple(value.shape)) for path, value in new_params.flat_state()]

There is no `strict=False` to lean on: the merge is yours to write, and so is
the report. Take from the file every entry whose name and shape match the new
model, keep the fresh initialization for the rest, and compute the two key sets
that say what happened: *missing*, parameters the model has but the file did not
fill, and *unexpected*, file entries with no home in the model.

In [ ]:
file_flat = load_file('mlp-jax.safetensors')
new_flat = flatten(new_params)
matched = {k: v for k, v in file_flat.items()
           if k in new_flat and v.shape == new_flat[k].shape}
merged = unflatten({**new_flat, **matched})
print('missing:', sorted(set(new_flat) - set(matched)))
print('unexpected:', sorted(set(file_flat) - set(new_flat)))
nnx.replace_by_pure_dict(new_params, merged)
nnx.update(classifier, new_params)
print('loaded trunk:', jnp.array_equal(
    classifier.hidden.kernel, restored['hidden']['kernel']))

Read both sets before trusting the merged tree. The two `head` entries under
*missing* are expected: that layer is new and meant to start random. The two
`output` entries under *unexpected* are the file's old head, stranded by the
rename, and a renamed layer is what *unexpected* usually means in practice. The
lesson is the same as with a built-in report, only stricter, because nothing
prints it unless you do: name which keys you expect in each set and treat
anything else as a bug.

In [ ]:
def labels(params):
    return jax.tree_util.tree_map_with_path(
        lambda path, _: ('train' if any(
            getattr(p, 'key', None) == 'head' for p in path) else 'freeze'),
        params)

tx = optax.multi_transform(
    {'train': optax.adam(0.05), 'freeze': optax.set_to_zero()}, labels)
optimizer = nnx.Optimizer(classifier, tx, wrt=nnx.Param)

hidden_before = classifier.hidden.kernel[...].copy()
head_before = classifier.head.kernel[...].copy()

@nnx.jit
def frozen_step(model, optimizer, X):
    loss, grads = nnx.value_and_grad(
        lambda m: jnp.square(m(X)).mean())(model)
    optimizer.update(model, grads)
    return loss

_ = frozen_step(classifier, optimizer, X)
print('trunk unchanged:', jnp.array_equal(
    hidden_before, classifier.hidden.kernel))
print('head changed:', not jnp.array_equal(head_before, classifier.head.kernel))

sizes = flatten(jax.tree_util.tree_map(jnp.size, merged))
flat_labels = flatten(labels(merged))
trainable = sum(int(s) for k, s in sizes.items() if flat_labels[k] == 'train')
total = sum(int(s) for s in sizes.values())
f'{trainable} trainable of {total}'

The Hugging Face Hub distributes JAX weights as safetensors, so the flat dict
you just merged has the same shape as the artifact you will download in
practice. This section covers *how* to load and adapt pretrained weights;
that section covers when it helps and how far to unfreeze.

## Summary

A saved model is state, not code: a pytree of named arrays that means something
only once the code that built the network runs again. For your own files
`jnp.save` is fine; for files you share, safetensors stores the same tensors
behind a plain JSON header, with no pickle to execute, which is why hubs
standardize on it. A resumable checkpoint contains model state, optimizer
state, and step in pytrees that Orbax saves atomically in a single call, or a
resume restarts the optimizer's momentum from zero. Loading someone else's
weights is pytree surgery, and the missing/unexpected sets you compute are a
diagnostic to read rather than a formality to skip.

## Exercises

1. Even if you never deploy to another machine, name two reasons to checkpoint.
   Then consider the atomic write: if the checkpoint were written straight to its
   final path (delete the `os.replace` from `save_checkpoint`, the
   rename-into-place that orbax performs, or the fresh numbered files that
   `tf.train.Checkpoint.save` writes), describe the failure a crash mid-write
   now causes, and why the atomic version avoids it.
1. Read the first 8 bytes of the safetensors file you wrote for the MLP as a
   little-endian integer, as the header cell does. How large is the JSON header
   for the MLP, and how does it grow if you double the hidden width?
1. Save the MLP's parameters cast to `bfloat16` and load them back into a
   `float32` model (that section). What is lost? Is that acceptable
   for inference? For resuming training?
1. Take two checkpoints of the regressor 50 steps apart, average their weight
   tensors into a third set of parameters, and evaluate it. The result previews
   weight averaging.